<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Transform with Python

**[Transform with Python](https://learning.nextia-ai.com/courses/sql/m04/transform-with-python/)** · SQL and Data Preparation for AI · Module 4, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** load the extract in pandas so that every column has the type that you chose, parse dates with an explicit format, join tables with a key rule that pandas checks, and change values in named steps that count what they change. The notebook shows the pandas 3 behaviour: the `str` type and copy-on-write.

| | |
|---|---|
| **Time** | About 50 minutes |
| **Needs** | An internet connection for the setup cells. No account and no local install: the first cell installs pandas 3.0.6 in this notebook. |
| **You should know** | What a profile found in the extract, from [Profile before changing](https://learning.nextia-ai.com/courses/sql/m04/profile-before-changing/). That a CSV file has no types, from [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/). |
| **You do not need** | The local project. The setup below builds the database and the extract here. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m04/transform-with-python/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M04-L02-transform-with-python/transform-with-python-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `pandas` 3.0.6 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6
import pandas; print("pandas", pandas.__version__)

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: rebuild the earlier lessons' files

This lesson starts from files that you made in earlier lessons: the extract `data/extract/` from [Reproducible extraction](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/). The next cell downloads the course's finished scripts ([C04/pipeline](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C04/pipeline)) into `ticket-data` and runs them in order, so you have the same files here. It takes about 20 seconds.

You should see one line per step, ending with `extract.py: done`. On your computer, your own scripts from those lessons make the same files.

In [ ]:
# Setup: rebuild the files of the earlier lessons with the course's scripts.
STEPS = ["extract"]
for file in ["queries/extract_tickets.sql", "queries/extract_outcomes.sql"] + [f"scripts/{s}.py" for s in STEPS]:
    if not Path(file).exists():
        download(LABS + "pipeline/" + file, Path(file))
for step in STEPS:
    result = subprocess.run([sys.executable, f"scripts/{step}.py"], capture_output=True, text=True)
    if result.returncode:
        raise SystemExit(f"{step}.py failed:\n{result.stdout}{result.stderr}")
    print(f"{step}.py: done")

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. pandas guesses types

A CSV file is text. When `pd.read_csv` reads it, pandas looks at the values in each column and **guesses** a type. Some guesses are wrong, and they cause no error.

The next cell reads three made-up tickets. Ticket T-2 has no priority, and its region is `NA`, a new code for North America. `io.StringIO(TINY)` lets `read_csv` read the text as if it were a file.

> **Predict.** What type does `priority` get? Then run the cell.

In [ ]:
import io

TINY = """ticket_id,priority,region,created_at
T-1,1,north,2026-03-04 09:15:00
T-2,,NA,2026-03-04 10:02:00
T-3,3,,2026-03-05 16:40:00
"""

guessed = pd.read_csv(io.StringIO(TINY))
print(guessed)
print(guessed.dtypes)

> **Check.** You should see three quiet changes:
>
> 1. `priority` is `float64`: 1.0 and 3.0. The standard integer type cannot hold a missing value, so pandas chose a type that can (`NaN`).
> 2. The region `NA` of T-2 became `NaN`. pandas treats some texts as missing by default: `NA`, `N/A`, `null`, `None`, `nan` and others. Now T-2 looks the same as T-3, which has no region.
> 3. `created_at` is `str`, text, not a date.

## 2. Load with types that you choose

The fix is to give the type of each column. Run the cell.

In [ ]:
typed = pd.read_csv(
    io.StringIO(TINY),
    dtype={"ticket_id": "string", "priority": "Int64", "region": "string"},
    keep_default_na=False,
    na_values=[""],
)
typed["created_at"] = pd.to_datetime(typed["created_at"], format="%Y-%m-%d %H:%M:%S")
print(typed)
print(typed.dtypes)

> **Check.** You should see `priority` as `Int64` with `1`, `<NA>` and `3`, the region `NA` kept as text, and `created_at` as `datetime64[us]`.

| Setting | What it does |
|---|---|
| `dtype={...}` | Gives the type of each column that you name. pandas does not guess for those columns. |
| `"Int64"` (capital I) | An integer type that can hold a missing value, shown as `<NA>`. Lowercase `int64` cannot. `"Float64"` is the same idea for decimal numbers. |
| `"string"` | A text type whose missing value is `<NA>`. |
| `keep_default_na=False` | Turns off the default list of missing-value texts. `NA` stays the text `NA`. |
| `na_values=[""]` | Only an empty cell is missing. In this export, an empty cell is how SQL's NULL arrives in a CSV file. |
| `pd.to_datetime(..., format=...)` | Turns text into a date and time, with the format that you give. |

**pandas 3: `str` and `string`.** A text column that pandas guesses gets the type `str`, and its missing value is `NaN`. The type `"string"`, which you ask for by name, marks a missing value as `<NA>`. They behave differently in comparisons, as you see in section 6.

## 3. Dates: always give the format

The **format** says where each part of a date is: `%Y` is the 4-digit year, `%m` the month, `%d` the day, and `%H:%M:%S` the time. Mei swaps the day and the month by mistake.

> **Predict.** What does the wrong format do with the first two values? Then run the cell.

In [ ]:
times = pd.Series(["2026-03-04 09:15:00", "2026-03-05 16:40:00", "2026-03-13 08:05:00"])

print(pd.to_datetime(times, format="%Y-%m-%d %H:%M:%S"))
print(pd.to_datetime(times[:2], format="%Y-%d-%m %H:%M:%S"))
try:
    print(pd.to_datetime(times, format="%Y-%d-%m %H:%M:%S"))
except ValueError as error:
    print("ValueError:", str(error).split(". You might")[0])

> **Check.** You should see three results:
>
> - With the correct format, every value is correct.
> - With the wrong format and the first two values, 4 March becomes **3 April** (`2026-04-03`), with no error. Every day is 12 or less, so every value is a possible date.
> - With the third value, day 13 cannot be a month, so pandas stops with `ValueError: time data "2026-03-13 08:05:00" doesn't match format`. That error is good news: it shows the mistake.

Without `format`, pandas guesses the format. For short dates such as `04/03/2026`, the guess is not safe: pandas 3.0.6 reads it as 3 April, month first. In the United Kingdom, the same text means 4 March. Only the source of the data can tell you which one is correct.

> **Your turn.** The dates in the next cell come from a UK system: day, month, year. The format in the cell is month first. Run the cell as it is: it gives no error, and both dates are wrong (3 April and 12 November). Then change `uk_format`, run the cell again, and run the check cell.

In [ ]:
uk = pd.Series(["04/03/2026", "11/12/2026"])  # day/month/year, as in the United Kingdom
uk_format = "%m/%d/%Y"  # month first: wrong for this source. Change it.
parsed = pd.to_datetime(uk, format=uk_format)
print(parsed)

In [ ]:
# Check: the first date must be 4 March 2026.
try:
    first = pd.to_datetime(uk, format=uk_format)[0]
    expect("the first date", first.strftime("%d %B %Y"), "04 March 2026")
except ValueError as error:
    print(f"Not yet: the format does not match the dates: {error}".splitlines()[0])

> **Warning.** `pd.to_datetime(..., errors="coerce")` does not stop at a bad value. It turns it into `NaT` ("not a time"), the missing value for dates. If you use it, count the `NaT` values before and after, and report the new ones.

## 4. Worked example: Mei loads the extract

Mei starts `scripts/clean.py`. In lesson 3 it gets the cleaning steps. Today it loads the extract with a type for every column. `json`, `OUT` and `REPORTS` are not used yet, and `main()` is a temporary check that lesson 3 replaces.

Run the cell. It saves the script in the project and prints `Writing scripts/clean.py`.

In [ ]:
%%writefile scripts/clean.py
"""Clean the extracted tickets and outcomes. Every input row ends in exactly
one place: the clean table, the quarantine table, or the count of removed
exact duplicates. The original values stay next to the corrected ones."""

import json
from pathlib import Path

import pandas as pd

IN = Path("data/extract")
OUT = Path("data/clean")
REPORTS = Path("reports")

TICKET_TYPES = {
    "ticket_id": "string", "customer_id": "string", "channel": "string", "team": "string",
    "priority": "Int64", "order_value": "Float64", "word_count": "Int64",
    "first_reply_minutes": "Int64", "priority_now": "Int64",
    "segment": "string", "region": "string",
}
TIME_FORMAT = "%Y-%m-%d %H:%M:%S"


def load_tickets():
    df = pd.read_csv(IN / "tickets.csv", dtype=TICKET_TYPES, keep_default_na=False, na_values=[""])
    df["created_at"] = pd.to_datetime(df["created_at"], format=TIME_FORMAT)
    df["closed_at"] = pd.to_datetime(df["closed_at"], format=TIME_FORMAT)
    df["joined_on"] = pd.to_datetime(df["joined_on"], format="%Y-%m-%d")
    return df


def main():
    tickets = load_tickets()
    print(f"Loaded {len(tickets)} ticket rows")
    print(tickets.dtypes)
    print("created_at:", tickets["created_at"].min(), "to", tickets["created_at"].max())
    print("missing closed_at (open tickets):", tickets["closed_at"].isna().sum())


if __name__ == "__main__":
    main()

> **Predict.** How many rows does the script load, and how many `closed_at` values are missing? Then run the script.

In [ ]:
!"{sys.executable}" scripts/clean.py

> **Check.** You should see `Loaded 1416 ticket rows`, every column with the type from `TICKET_TYPES` (`string`, `Int64`, `Float64`, and `datetime64[us]` for the three dates), `created_at: 2026-01-05 07:30:24 to 2026-06-30 20:04:36`, and `missing closed_at (open tickets): 12`.

Mei checks three things against the profile: 1,416 rows, the same as the extract; the `created_at` range stays inside 1 January to 30 June; and 12 `closed_at` values are missing, the same 12 open tickets. So `pd.to_datetime` did not create a new missing value.

## 5. Join tables in pandas, with a key rule

In pandas, a join is `merge`. Mei repeats the extract's customer join from the raw files, to see two settings that SQL does not have. `**READ` passes the two settings in the dictionary `READ` to both `read_csv` calls, so they cannot differ. Run the cell to save the script.

In [ ]:
%%writefile scripts/check_join.py
"""Repeat the extract's customer join in pandas, and count every row."""

import pandas as pd

READ = {"keep_default_na": False, "na_values": [""]}
tickets = pd.read_csv("data/raw/tickets.csv", dtype={"customer_id": "string"}, **READ)
customers = pd.read_csv("data/raw/customers.csv", dtype="string", **READ)
print(f"tickets: {len(tickets)} rows, customers: {len(customers)} rows")

joined = tickets.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
print(f"joined: {len(joined)} rows")
print(joined["_merge"].value_counts())

no_match = joined[joined["_merge"] == "left_only"]
print(f"no customer_id: {no_match['customer_id'].isna().sum()}")
print(f"customer_id not in customers: {no_match['customer_id'].notna().sum()}")

> **Predict.** `tickets` has 1,417 rows. How many rows does the left join return? Then run the script.

In [ ]:
!"{sys.executable}" scripts/check_join.py

> **Check.** You should see `joined: 1417 rows`, then `both 1396`, `left_only 21` and `right_only 0`, then `no customer_id: 12` and `customer_id not in customers: 9`. The counts agree with Module 2.

| Setting | What it does |
|---|---|
| `how="left"` | Keeps every row of the left table, as `LEFT JOIN` does. |
| `validate="many_to_one"` | A rule: each `customer_id` must occur **once** in `customers`. If not, `merge` stops with an error, before rows multiply. |
| `indicator=True` | Adds the column `_merge`: `both`, `left_only` or `right_only`. |

The key rule is the important part. Mei now joins the outcome events to the extract's tickets, to add the `team` to each event. Each event belongs to one ticket, so the rule is `many_to_one`.

> **Predict.** The extract has 1,869 events. How many rows does the join return without the rule? Then run the cell.

In [ ]:
tickets_x = pd.read_csv("data/extract/tickets.csv", dtype="string", keep_default_na=False, na_values=[""])
events_x = pd.read_csv("data/extract/outcomes.csv", dtype="string", keep_default_na=False, na_values=[""])
print("events:", len(events_x))
print("joined without the rule:", len(events_x.merge(tickets_x[["ticket_id", "team"]], on="ticket_id", how="left")))
try:
    events_x.merge(tickets_x[["ticket_id", "team"]], on="ticket_id", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print("MergeError:", str(error)[:220])

> **Check.** You should see 1,892 rows without the rule: 23 events now appear twice, because their ticket appears twice in the extract. Nothing fails. With the rule, pandas stops with `MergeError: Merge keys are not unique in right dataset; not a many-to-one merge`, and names the duplicated keys, starting with `T-10031`. These are the 14 exact duplicates from the profile. After `drop_duplicates()`, the rule holds. You try that in section 8.

## 6. Change values in named steps

A **transformation** is a change to the values of a column. Mei follows four rules for each one:

1. **One named step** for one problem, with a comment that says why.
2. **Keep the original** in a column next to it, for example `channel_raw`.
3. **Count** the rows that the step changes.
4. **Change rows with `.loc`**, in one statement.

Here is the channel respelling from the profile, in a short script. `.str.strip()` removes spaces at the start and the end, `.str.lower()` makes all letters small, and `.replace(...)` changes the one wrong word. Run the cell to save the script.

In [ ]:
%%writefile scripts/try_channel.py
import pandas as pd

from clean import load_tickets

tickets = load_tickets().drop_duplicates()
print(f"rows: {len(tickets)}")

channel = tickets["channel"].str.strip().str.lower().replace({"webform": "web_form"})
changed = channel != tickets["channel"]
print(changed.value_counts(dropna=False))
print(f"respelled: {changed.fillna(False).sum()}")

tickets["channel_raw"] = tickets["channel"]
tickets["channel"] = channel
print(tickets.loc[changed.fillna(False), ["ticket_id", "channel_raw", "channel"]].drop_duplicates("channel_raw"))
print(tickets["channel"].value_counts(dropna=False))

> **Predict.** The profile found 40 rows with other spellings. How many rows does this step respell? Then run the script.

In [ ]:
!"{sys.executable}" scripts/try_channel.py

> **Check.** You should see `rows: 1402`, then `False 1325`, `True 39` and `<NA> 38`, `respelled: 39`, four example rows, and the new counts: `email 564`, `chat 415`, `web_form 203`, `phone 182`, `<NA> 38`.

- 1,402 rows: the 14 exact duplicates are gone.
- 39, not 40: one `EMAIL` row was one of the duplicates.
- In row `T-10124`, `channel_raw` is ` chat`, with the space at the start. The table does not show the space, but the comparison does.
- 38 rows are `<NA>`: a missing channel stays missing. With the `"string"` type, a comparison with a missing value gives `<NA>`, not `True` or `False`. `fillna(False)` says "a missing value did not change".

### Change rows with .loc, not in two steps

pandas 3 uses **copy-on-write**: when you select rows, you get a new table that behaves as a copy. A change to the copy never changes the original table. Run the cell to save the script `cow.py`.

In [ ]:
%%writefile cow.py
import pandas as pd

df = pd.DataFrame({
    "ticket_id": ["T-1", "T-2", "T-3"],
    "team": ["account", "delivery", "returns"],
    "order_value": [None, 40.0, -1.0],
})

df[df["order_value"] == -1]["order_value"] = None
print(df)

df.loc[df["order_value"] == -1, "order_value"] = None
print(df)

> **Predict.** The script sets the `-1` of T-3 to missing twice: first in two steps, then with `.loc`. Which one changes `df`? Then run the script.

In [ ]:
!"{sys.executable}" cow.py

> **Check.** You should see a `ChainedAssignmentError` warning first. Then the first table still has `-1.0` for T-3, and the second table has `NaN`.

The first form selects the rows, then sets a column on that selection: a **chained assignment**. In pandas 3 it never changes `df`. pandas prints a warning, but the program continues. The `.loc[rows, column] = value` form does the change in one step, on `df` itself. Code from older tutorials can do nothing in pandas 3.

## 7. Guided practice: the default `str` type

The next script does the same respelling, but loads the file with no `dtype`, so `channel` gets the guessed type. Run the cell to save it.

In [ ]:
%%writefile scripts/try_str.py
import pandas as pd

plain = pd.read_csv("data/extract/tickets.csv").drop_duplicates()
print(plain["channel"].dtype)
respelled = plain["channel"].str.strip().str.lower().replace({"webform": "web_form"})
print("changed:", (respelled != plain["channel"]).sum())

> **Predict.** With the `"string"` type, 39 rows changed. How many does this script count? Write it down, then run the script.

In [ ]:
!"{sys.executable}" scripts/try_str.py

> **Check.** You should see `str` and `changed: 77`.

77 = 39 respelled rows + 38 missing channels. With the guessed `str` type, a missing value is `NaN`, and `NaN != NaN` is `True`. So every missing channel counts as "changed". Both scripts run without an error; only the count shows the difference.

On your computer, add this result to `reports/profile.md` under "Categories": 39 rows respelled after duplicates are removed.

## 8. Your turn: load and join the outcome events

> **Your turn.** In the next cell:
>
> 1. Load `data/extract/outcomes.csv` with types: `csat` as `"Int64"`, the other columns as `"string"`. Only an empty cell is missing.
> 2. Parse `recorded_at` as a date with `TIME_FORMAT`.
> 3. Join the events to the deduplicated tickets (`ticket_id` and `team` only) with `how="left"`, `validate="many_to_one"` and `indicator=True`.
>
> Predict the number of rows and the `_merge` counts first. Then run the cell and the check cell.

In [ ]:
sys.path.insert(0, "scripts")  # so that Python finds clean.py
from clean import IN, TIME_FORMAT, load_tickets

# 1. Load data/extract/outcomes.csv: csat as "Int64", the other columns as "string".
#    Only an empty cell is missing.
events = ...

# 2. Parse recorded_at with TIME_FORMAT.


# 3. Join the events to the deduplicated tickets (only ticket_id and team),
#    with how="left", validate="many_to_one" and indicator=True.
tickets = load_tickets().drop_duplicates()
joined = ...

In [ ]:
# Check: the types of events, and the rows of joined.
if not (isinstance(events, pd.DataFrame) and isinstance(joined, pd.DataFrame)):
    print("Not yet: events and joined must be DataFrames. Replace each ... with your code.")
else:
    expect("the type of csat", str(events["csat"].dtype), "Int64")
    expect("the type of status", str(events["status"].dtype), "string")
    expect("the type of recorded_at", str(events["recorded_at"].dtype), "datetime64[us]")
    expect("the number of joined rows", len(joined), len(events))
    expect("the indicator column", "_merge" if "_merge" in joined.columns else "missing", "_merge")

When the check passes, the rule holds after `drop_duplicates()`: 1,869 events in, 1,869 rows out, and every event found its ticket (`both 1869`). The extract left out the 4 orphan events in SQL, so `left_only` is 0.

## 9. Failure case: the dates look correct in a sample

Mei tests a date format on the first 50 rows of the extract. There is no error.

> **Predict.** The export ends in June. Is the format correct? Then run the cell.

In [ ]:
sample = pd.read_csv("data/extract/tickets.csv", nrows=50)
created = pd.to_datetime(sample["created_at"], format="%Y-%d-%m %H:%M:%S")
print(sample["created_at"].iloc[0], "->", created.iloc[0])
print("range:", created.min(), "to", created.max())

> **Check.** You should see `2026-01-05 07:30:24 -> 2026-05-01 07:30:24`, and a range from May to **October**. But the export ends in June.

**Cause:** the format has the day and the month in the wrong places. The first 50 tickets are from 5 to 12 January, so every day is 12 or less, and every value is a possible date. Run the cell to parse the full column with the same format.

In [ ]:
full = pd.read_csv("data/extract/tickets.csv")
try:
    pd.to_datetime(full["created_at"], format="%Y-%d-%m %H:%M:%S")
except ValueError as error:
    print("ValueError:", str(error).split(". You might")[0])

> **Check.** You should see `ValueError: time data "2026-01-13 09:57:09" doesn't match format`, at the first day after the 12th.

**Fix:** write the format from the data, part by part: `2026-01-05` is year, month, day, so `%Y-%m-%d`. Parse the full column, not a sample. After parsing, check the smallest and largest date against the profile.

**In AI work:** the date decides which rows go into training and which into testing (Module 5). Wrong dates put January tickets in October, so the model trains on rows that look like they come from the future.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m04/transform-with-python/). They count toward your certificate when you are signed in and enrolled.

In this lesson you gave every column a type with `dtype=`, kept the text `NA` with `keep_default_na=False`, parsed dates with an explicit format and checked their range, joined with `validate="many_to_one"` and `indicator=True`, and changed values in named, counted steps with `.loc`.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Handle messy values](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m04/transform-with-python/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.